In [ ]:
import os
import numpy as np
import cv2
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# ── 1. Load images ──────────────────────────────────────────
def load_fer2013(root, size=(48, 48)):
    X, y = [], []
    for emotion in os.listdir(root):
        folder = os.path.join(root, emotion)
        if not os.path.isdir(folder):
            continue
        for fname in os.listdir(folder):
            img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, size)
            X.append(img.flatten().astype(np.float64))
            y.append(emotion)
    return np.array(X), np.array(y)

X_train, y_train = load_fer2013("../datasets/fer2013/train")
X_test,  y_test  = load_fer2013("../datasets/fer2013/test")

# Normalise pixel values to [0, 1]
X_train /= 255.0
X_test  /= 255.0

# Encode string labels → integers
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test  = le.transform(y_test)

# ── 2. PCA (eigenfaces) ──────────────────────────────────────
n_components = 80  # try 50–150
pca = PCA(n_components=n_components, whiten=True)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.1%}")

# ── 3. Train SVM classifier ──────────────────────────────────
clf = SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')
clf.fit(X_train_pca, y_train)

# ── 4. Evaluate ──────────────────────────────────────────────
y_pred = clf.predict(X_test_pca)
print(classification_report(y_test, y_pred, target_names=le.classes_))

Explained variance: 88.2%
              precision    recall  f1-score   support

       angry       0.34      0.38      0.36       958
     disgust       0.72      0.44      0.55       111
        fear       0.36      0.37      0.36      1024
       happy       0.61      0.65      0.63      1774
     neutral       0.46      0.46      0.46      1233
         sad       0.40      0.37      0.39      1247
    surprise       0.73      0.63      0.67       831

    accuracy                           0.49      7178
   macro avg       0.52      0.47      0.49      7178
weighted avg       0.49      0.49      0.49      7178



In [15]:
import joblib

name = "meow"

# After training, save both objects
joblib.dump(pca, f'models/eigenface_pca_{name}.pkl')
joblib.dump(clf, f'models/eigenface_clf_{name}.pkl')
joblib.dump(le,  f'models/eigenface_labels_{name}.pkl')  # save this too for inverse_transform

print("Models saved.")

Models saved.


# LATER ON (THIS IS IN THE EXEC)

In [20]:
img = "../image.png"

In [21]:
import joblib

pca = joblib.load(f'models/eigenface_pca_{name}.pkl')
clf = joblib.load(f'models/eigenface_clf_{name}.pkl')
le  = joblib.load(f'models/eigenface_labels_{name}.pkl')

# predict_emotion() will now work exactly as before
print(predict_emotion(img))

fear
